# DPO-SDXL × DrawBench — ImageReward Scoring (Colab)

In [1]:
# Pin transformers (v4 — ImageReward's BLIP needs helpers removed in v5)
# and diffusers (ImageReward's ReFL.py imports diffusers at module level).
!pip install -q transformers==4.44.2 diffusers==0.30.3 accelerate torch torchvision pillow pandas
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip uninstall -y wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 94.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 130.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     | 4.3 MB 8.2 MB/s 0:00:000m0m
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 9.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
Found existing installation: wandb 0.27.2
Uninstalling wandb-0.27.2:
  Successfully uninstalled wandb

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


## 1. Data Preparation

Same approacj as the main evaluation notebook: copy the zips from Drive, unzip locally.

In [6]:
DRIVE_DIR = Path("/content/drive/MyDrive/generative-modelling")

DATA_DIR = Path("/content/data")
STANDARD_DIR = DATA_DIR / "dpo_sdxl_drawbench"
ZSAMPLING_DIR = DATA_DIR / "dpo_sdxl_drawbench_zsampling"

SEEDS = [42, 123, 2026]
NUM_PROMPTS = 200

In [7]:
import time

DATA_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()
for name in ["dpo_sdxl_drawbench.zip", "dpo_sdxl_drawbench_zsampling.zip"]:
    target = DATA_DIR / name.replace(".zip", "")
    if not target.exists():
        src = DRIVE_DIR / name
        print(f"Extracting {name} directly from Drive...")
        with zipfile.ZipFile(src) as z:
            z.extractall(DATA_DIR)
    else:
        print(f"{name.replace('.zip', '')}/ already extracted.")

print(f"Done in {time.time() - t0:.1f}s")
print(f"Standard: {STANDARD_DIR}")
print(f"Z-Sampling: {ZSAMPLING_DIR}")

Extracting dpo_sdxl_drawbench.zip directly from Drive...
Extracting dpo_sdxl_drawbench_zsampling.zip directly from Drive...
Done in 40.0s
Standard: /content/data/dpo_sdxl_drawbench
Z-Sampling: /content/data/dpo_sdxl_drawbench_zsampling


In [8]:
def index_images(base_dir: Path) -> dict[str, Path]:
    """Build a filename -> path index across all checkpoint subfolders."""
    index = {}
    for png in base_dir.rglob("*.png"):
        index[png.name] = png
    return index

std_index = index_images(STANDARD_DIR)
zs_index = index_images(ZSAMPLING_DIR)

print(f"Standard images found: {len(std_index)} (expected 600)")
print(f"Z-Sampling images found: {len(zs_index)} (expected 600)")
assert len(std_index) == 600, f"Expected 600 standard images, got {len(std_index)}"
assert len(zs_index) == 600, f"Expected 600 z-sampling images, got {len(zs_index)}"

Standard images found: 600 (expected 600)
Z-Sampling images found: 600 (expected 600)


## 2. Load DrawBench Prompts & Build Pairs

In [9]:
CSV_URL = "https://huggingface.co/datasets/sayakpaul/drawbench/raw/main/DrawBench%20Prompts%20-%20Sheet1.csv"
df_prompts = pd.read_csv(CSV_URL)
PROMPTS = df_prompts["Prompts"].tolist()
assert len(PROMPTS) == NUM_PROMPTS, f"Expected {NUM_PROMPTS} prompts, got {len(PROMPTS)}"
print(f"Loaded {len(PROMPTS)} DrawBench prompts")

Loaded 200 DrawBench prompts


In [10]:
def build_pairs() -> pd.DataFrame:
    """Build a DataFrame of (prompt_idx, seed, prompt, standard_path, zsampling_path)."""
    rows = []
    missing = []
    for seed in SEEDS:
        for idx in range(NUM_PROMPTS):
            std_name = f"dpo_sdxl_prompt{idx:03d}_seed{seed}.png"
            zs_name = f"dpo_sdxl_zsample_prompt{idx:03d}_seed{seed}.png"
            std_path = std_index.get(std_name)
            zs_path = zs_index.get(zs_name)
            if std_path is None or zs_path is None:
                missing.append((idx, seed, std_name if std_path is None else zs_name))
                continue
            rows.append({
                "prompt_idx": idx,
                "seed": seed,
                "prompt": PROMPTS[idx],
                "standard_path": str(std_path),
                "zsampling_path": str(zs_path),
            })
    if missing:
        print(f"WARNING: {len(missing)} missing images!")
        for idx, seed, name in missing[:5]:
            print(f"  prompt={idx} seed={seed}: {name}")
    else:
        print("All 600 pairs verified.")
    return pd.DataFrame(rows)

df_pairs = build_pairs()
print(f"Total pairs: {len(df_pairs)}")

All 600 pairs verified.
Total pairs: 600


## 3. Load ImageReward

In [11]:
import ImageReward as RM

ir_model = RM.load("ImageReward-v1.0", device=DEVICE)
print("ImageReward model loaded.")

ImageReward.pt:   0%|          | 0.00/1.79G [00:00<?, ?B/s]

load checkpoint from /root/.cache/ImageReward/ImageReward.pt


med_config.json:   0%|          | 0.00/485 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

checkpoint loaded
ImageReward model loaded.


## 4. Score All 600 Pairs

In [12]:
from tqdm.auto import tqdm

ir_results = []
wins_zs = 0
t0 = time.time()

bar = tqdm(df_pairs.iterrows(), total=len(df_pairs), desc="ImageReward", unit="pair")
for i, (_, row) in enumerate(bar):
    prompt = row["prompt"]
    ir_std = float(ir_model.score(prompt, row["standard_path"]))
    ir_zs  = float(ir_model.score(prompt, row["zsampling_path"]))

    ir_results.append({
        "prompt_idx": int(row["prompt_idx"]),
        "seed": int(row["seed"]),
        "ir_standard": ir_std,
        "ir_zsampling": ir_zs,
    })

    if ir_zs > ir_std:
        wins_zs += 1
    bar.set_postfix(zs_win=f"{wins_zs/(i+1)*100:.0f}%")

    if (i + 1) % 50 == 0:
        tqdm.write(
            f"  [{i+1:>4}/{len(df_pairs)}] "
            f"last: std={ir_std:.3f} zs={ir_zs:.3f} | "
            f"Z-win rate so far: {wins_zs/(i+1)*100:.1f}%"
        )

elapsed = time.time() - t0
print(f"\nDone. Scored {len(df_pairs)} pairs in {elapsed/60:.1f} min")

df_ir = pd.DataFrame(ir_results)
print(f"IR mean — Standard: {df_ir['ir_standard'].mean():.4f}, Z-Sampling: {df_ir['ir_zsampling'].mean():.4f}")
print(f"Z-Sampling wins: {wins_zs}/{len(df_pairs)} ({wins_zs/len(df_pairs)*100:.1f}%)")

ImageReward:   0%|          | 0/600 [00:00<?, ?pair/s]

  [  50/600] last: std=1.691 zs=1.332 | Z-win rate so far: 50.0%
  [ 100/600] last: std=-0.755 zs=-0.817 | Z-win rate so far: 51.0%
  [ 150/600] last: std=1.734 zs=1.806 | Z-win rate so far: 52.0%
  [ 200/600] last: std=1.916 zs=1.902 | Z-win rate so far: 52.5%
  [ 250/600] last: std=0.936 zs=1.277 | Z-win rate so far: 52.8%
  [ 300/600] last: std=0.512 zs=1.802 | Z-win rate so far: 54.0%
  [ 350/600] last: std=1.870 zs=1.655 | Z-win rate so far: 52.0%
  [ 400/600] last: std=1.616 zs=1.661 | Z-win rate so far: 52.8%
  [ 450/600] last: std=1.581 zs=1.686 | Z-win rate so far: 53.1%
  [ 500/600] last: std=-0.269 zs=-0.515 | Z-win rate so far: 52.2%
  [ 550/600] last: std=1.925 zs=1.838 | Z-win rate so far: 53.1%
  [ 600/600] last: std=1.271 zs=0.865 | Z-win rate so far: 52.7%

Done. Scored 600 pairs in 2.3 min
IR mean — Standard: 0.8903, Z-Sampling: 0.9249
Z-Sampling wins: 316/600 (52.7%)


## 5. Merge IR Scores

In [13]:
import json

RESULTS_PATH = DRIVE_DIR / "dpo_sdxl_drawbench" / "dpo_sdxl_drawbench_eval_results.jsonl"
assert RESULTS_PATH.exists(), f"Results file not found: {RESULTS_PATH}"

with open(RESULTS_PATH) as f:
    records = [json.loads(line) for line in f if line.strip()]
print(f"Loaded {len(records)} existing records from JSONL.")

ir_lookup = {(r["prompt_idx"], r["seed"]): r for r in ir_results}

updated = 0
for rec in records:
    key = (rec["prompt_idx"], rec["seed"])
    if key in ir_lookup:
        rec["ir_standard"] = ir_lookup[key]["ir_standard"]
        rec["ir_zsampling"] = ir_lookup[key]["ir_zsampling"]
        updated += 1

with open(RESULTS_PATH, "w") as f:
    for rec in records:
        f.write(json.dumps(rec) + "\n")

os.sync()
print(f"Updated {updated}/{len(records)} records with IR scores.")
print(f"Written to: {RESULTS_PATH}")

Loaded 600 existing records from JSONL.
Updated 600/600 records with IR scores.
Written to: /content/drive/MyDrive/generative-modelling/dpo_sdxl_drawbench/dpo_sdxl_drawbench_eval_results.jsonl


## 6. Summary

In [ ]:
print("=== ImageReward: DPO-SDXL Standard vs Z-Sampling ===")
print(f"Pairs scored: {len(df_ir)}")
print(f"")
std_mean = df_ir["ir_standard"].mean()
zs_mean = df_ir["ir_zsampling"].mean()
print(f"  Standard  (mean): {std_mean:.4f}")
print(f"  Z-Sampling (mean): {zs_mean:.4f}")
print(f"  Delta (Z - Std):   {zs_mean - std_mean:+.4f}")
print(f"")
print(f"  Z-Sampling wins: {wins_zs}/{len(df_ir)} ({wins_zs/len(df_ir)*100:.1f}%)")
print(f"")
print("Per-seed breakdown:")
for seed in SEEDS:
    sub = df_ir[df_ir["seed"] == seed]
    s_mean = sub["ir_standard"].mean()
    z_mean = sub["ir_zsampling"].mean()
    z_wins = (sub["ir_zsampling"] > sub["ir_standard"]).sum()
    print(f"  Seed {seed:>4}: Std={s_mean:.4f}  Z={z_mean:.4f}  Δ={z_mean-s_mean:+.4f}  Z-wins={z_wins}/{len(sub)}")